# 04 : Pandas

강의자료: `docs/course/04_Pandas.md`

Series/DataFrame 구조를 익힌 뒤 뉴욕 항공편 33만 행으로 선택·결측치·정렬·파생변수·
groupby·구조변환·결합까지 전처리 전 과정을 한다.

## 데이터

`data/raw/nycflights13.csv` — 2013년 뉴욕 3개 공항(EWR/LGA/JFK) 출발 항공편 336,776건.
수업 실습 저장소(`https://github.com/jslee-gses/code`)에서 가져왔다.
강의자료에 원 출처 URL이 없어 강사 확인이 필요하다.

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

## 1. Series — 값 + 인덱스

In [2]:
s = pd.Series([10, 20, 30, 40])
print(s)
print("\n값:", s.values)
print("인덱스:", list(s.index))

# 인덱스를 직접 지정할 수도 있다
s2 = pd.Series([947, 335, 238], index=["서울", "부산", "대구"])
print("\n", s2, sep="")
print("\n서울:", s2["서울"])

0    10
1    20
2    30
3    40
dtype: int64

값: [10 20 30 40]
인덱스: [0, 1, 2, 3]

서울    947
부산    335
대구    238
dtype: int64

서울: 947


## 2. DataFrame 만들기와 열 선택

In [3]:
data = {
    "도시": ["서울", "부산", "대구", "인천"],
    "인구": [947, 335, 238, 296],
    "면적": [605.2, 770.1, 883.5, 1063.3],
}
df = pd.DataFrame(data)
print(df)

print("\n대괄호 방식:\n", df["인구"], sep="")
print("\n점 방식:", list(df.인구))   # 공백·특수문자 없는 열 이름에서만 쓸 수 있다

   도시   인구      면적
0  서울  947   605.2
1  부산  335   770.1
2  대구  238   883.5
3  인천  296  1063.3

대괄호 방식:
0    947
1    335
2    238
3    296
Name: 인구, dtype: int64

점 방식: [947, 335, 238, 296]


## 3. 열 추가와 삭제

In [4]:
df["인구밀도"] = (df["인구"] * 10000 / df["면적"]).round(1)
df["권역"] = ["수도권", "영남", "영남", "수도권"]
print(df)

df2 = df.drop("권역", axis=1)      # 원본을 두고 사본을 만든다
print("\ndrop 후:\n", df2, sep="")

del df["권역"]                      # 원본에서 바로 지운다
print("\ndel 후 열:", list(df.columns))

   도시   인구      면적     인구밀도   권역
0  서울  947   605.2  15647.7  수도권
1  부산  335   770.1   4350.1   영남
2  대구  238   883.5   2693.8   영남
3  인천  296  1063.3   2783.8  수도권

drop 후:
   도시   인구      면적     인구밀도
0  서울  947   605.2  15647.7
1  부산  335   770.1   4350.1
2  대구  238   883.5   2693.8
3  인천  296  1063.3   2783.8

del 후 열: ['도시', '인구', '면적', '인구밀도']


## 4. loc와 iloc

`loc`은 **라벨**로, `iloc`은 **정수 위치**로 고른다.
`loc`의 슬라이싱은 끝을 포함하고, `iloc`은 포함하지 않는다.

In [5]:
print("loc[1] — 인덱스 라벨 1:\n", df.loc[1], sep="")
print("\niloc[0] — 첫 번째 행:\n", df.iloc[0], sep="")

print("\nloc[0:2] (끝 포함):\n", df.loc[0:2, ["도시", "인구"]], sep="")
print("\niloc[0:2] (끝 제외):\n", df.iloc[0:2, 0:2], sep="")

print("\n열 순서 바꾸기:\n", df[["인구", "도시", "면적", "인구밀도"]], sep="")
print("\n전치:\n", df.T, sep="")

loc[1] — 인덱스 라벨 1:
도시          부산
인구         335
면적       770.1
인구밀도    4350.1
Name: 1, dtype: object

iloc[0] — 첫 번째 행:
도시           서울
인구          947
면적        605.2
인구밀도    15647.7
Name: 0, dtype: object

loc[0:2] (끝 포함):
   도시   인구
0  서울  947
1  부산  335
2  대구  238

iloc[0:2] (끝 제외):
   도시   인구
0  서울  947
1  부산  335

열 순서 바꾸기:
    인구  도시      면적     인구밀도
0  947  서울   605.2  15647.7
1  335  부산   770.1   4350.1
2  238  대구   883.5   2693.8
3  296  인천  1063.3   2783.8

전치:
            0       1       2       3
도시         서울      부산      대구      인천
인구        947     335     238     296
면적      605.2   770.1   883.5  1063.3
인구밀도  15647.7  4350.1  2693.8  2783.8


## 5. 조건 필터링과 파생변수

In [6]:
# 여러 조건은 & | 로 묶고 각 조건을 괄호로 감싼다
big = df[(df["인구"] > 300) & (df["면적"] < 800)]
print("인구 300만 초과 & 면적 800 미만:\n", big, sep="")

# 조건 결과를 그대로 열로 저장할 수 있다
df["대도시"] = df["인구"] > 300
print("\n", df, sep="")

인구 300만 초과 & 면적 800 미만:
   도시   인구     면적     인구밀도
0  서울  947  605.2  15647.7
1  부산  335  770.1   4350.1

   도시   인구      면적     인구밀도    대도시
0  서울  947   605.2  15647.7   True
1  부산  335   770.1   4350.1   True
2  대구  238   883.5   2693.8  False
3  인천  296  1063.3   2783.8  False


## 6. 실제 데이터 불러오기

In [7]:
flights = pd.read_csv("data/raw/nycflights13.csv")
print("shape:", flights.shape)
print("\n열 목록:", list(flights.columns))
flights.head()

shape: (336776, 16)

열 목록: ['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay', 'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute']


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,origin,dest,air_time,distance,hour,minute
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,EWR,IAH,227.0,1400,5,15
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,LGA,IAH,227.0,1416,5,29
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,JFK,MIA,160.0,1089,5,40
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,JFK,BQN,183.0,1576,5,45
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,LGA,ATL,116.0,762,6,0


In [8]:
core = flights[["year", "month", "day", "hour", "origin", "dest",
                "carrier", "air_time", "distance", "dep_delay"]]
core.head()

,year,month,day,hour,origin,dest,carrier,air_time,distance,dep_delay
0,2013,1,1,5,EWR,IAH,UA,227.0,1400,2.0
1,2013,1,1,5,LGA,IAH,UA,227.0,1416,4.0
2,2013,1,1,5,JFK,MIA,AA,160.0,1089,2.0
3,2013,1,1,5,JFK,BQN,B6,183.0,1576,-1.0
4,2013,1,1,6,LGA,ATL,DL,116.0,762,-6.0


## 7. loc 복합 선택과 다중 조건

In [9]:
# 조건과 열을 한 번에 지정한다
sel = flights.loc[flights["month"] == 1, ["month", "day", "origin", "dest", "distance"]]
print("1월 항공편:", len(sel))
print(sel.head())

# AND
jfk_long = flights[(flights["origin"] == "JFK") & (flights["distance"] > 2000)]
print("\nJFK 출발 & 2000마일 초과:", len(jfk_long))

# OR
ewr_or_lga = flights[(flights["origin"] == "EWR") | (flights["origin"] == "LGA")]
print("EWR 또는 LGA 출발:", len(ewr_or_lga))

1월 항공편: 27004
   month  day origin dest  distance
0      1    1    EWR  IAH      1400
1      1    1    LGA  IAH      1416
2      1    1    JFK  MIA      1089
3      1    1    JFK  BQN      1576
4      1    1    LGA  ATL       762

JFK 출발 & 2000마일 초과: 32189
EWR 또는 LGA 출발: 225497


## 8. 결측치

In [10]:
na_counts = flights.isna().sum()
print("결측치가 있는 열:\n", na_counts[na_counts > 0], sep="")

print("\nair_time이 빈 행:", flights["air_time"].isna().sum())
print("air_time이 있는 행:", flights["air_time"].notna().sum())

# 채우기 — 다만 air_time을 0으로 채우면 "비행시간 0분"이라는 잘못된 값이 된다.
# 여기서는 방법만 확인하고 실제 분석에는 notna()로 걸러 쓴다.
filled = flights["air_time"].fillna(0)
print("0으로 채운 뒤 결측:", filled.isna().sum())

clean = flights[flights["air_time"].notna()].copy()
print("결측 제거 후:", clean.shape)

결측치가 있는 열:
dep_time     8255
dep_delay    8255
arr_time     8713
arr_delay    9430
air_time     9430
dtype: int64

air_time이 빈 행: 9430
air_time이 있는 행: 327346
0으로 채운 뒤 결측: 0
결측 제거 후: (327346, 16)


## 9. 정렬

In [11]:
print("거리 긴 순:")
print(flights.sort_values(by="distance", ascending=False)
      [["origin", "dest", "distance"]].head())

print("\n월·일 순:")
print(flights.sort_values(by=["month", "day"])[["month", "day", "origin"]].head())

거리 긴 순:


       origin dest  distance
50676     JFK  HNL      4983
108078    JFK  HNL      4983
100067    JFK  HNL      4983
179566    JFK  HNL      4983
30229     JFK  HNL      4983

월·일 순:
   month  day origin
0      1    1    EWR
1      1    1    LGA
2      1    1    JFK
3      1    1    JFK
4      1    1    LGA


## 10. 파생변수

In [12]:
f = flights.copy()

f["distance_km"] = (f["distance"] * 1.609344).round(1)   # 마일 → km
f["route"] = f["origin"] + "-" + f["dest"]               # 문자열 결합

print(f[["origin", "dest", "route", "distance", "distance_km"]].head())

  origin dest    route  distance  distance_km
0    EWR  IAH  EWR-IAH      1400       2253.1
1    LGA  IAH  LGA-IAH      1416       2278.8
2    JFK  MIA  JFK-MIA      1089       1752.6
3    JFK  BQN  JFK-BQN      1576       2536.3
4    LGA  ATL  LGA-ATL       762       1226.3


## 11. groupby

In [13]:
print("출발공항별 거리 통계:")
print(flights.groupby("origin")["distance"].agg(["count", "mean", "min", "max"]).round(1))

print("\n이름을 붙인 집계:")
print(flights.groupby("origin").agg(
    편수=("distance", "count"),
    평균거리=("distance", "mean"),
    최대거리=("distance", "max"),
).round(1))

print("\n다중 키 (출발공항 x 월) 상위 6개:")
print(flights.groupby(["origin", "month"])["distance"].mean().round(1).head(6))

print("\n열마다 다른 집계:")
print(flights.groupby("origin").agg({"distance": "mean", "air_time": "median",
                                     "carrier": "nunique"}).round(1))

출발공항별 거리 통계:


         count    mean  min   max
origin                           
EWR     120835  1056.7   17  4963
JFK     111279  1266.2   94  4983
LGA     104662   779.8   96  1620

이름을 붙인 집계:
            편수    평균거리  최대거리
origin                      
EWR     120835  1056.7  4963
JFK     111279  1266.2  4983
LGA     104662   779.8  1620

다중 키 (출발공항 x 월) 상위 6개:
origin  month
EWR     1         962.8
        2         958.1
        3         978.2
        4        1043.6
        5        1057.4
        6        1095.2
Name: distance, dtype: float64

열마다 다른 집계:


        distance  air_time  carrier
origin                             
EWR       1056.7     130.0       12
JFK       1266.2     149.0       10
LGA        779.8     115.0       13


## 12. 범주형 변수 만들기

In [14]:
f = flights.copy()

# np.select — 조건 목록과 값 목록을 짝지어 분류한다
conditions = [f["carrier"].isin(["UA", "AA", "DL"]),
              f["carrier"].isin(["B6", "EV", "MQ"])]
choices = ["A그룹", "B그룹"]
f["carrier_group"] = np.select(conditions, choices, default="기타")

print(f["carrier_group"].value_counts())

# loc로 특정 조건의 행만 수정
f.loc[f["carrier"] == "UA", "carrier_group"] = "A그룹(대형)"
print("\n수정 후:\n", f["carrier_group"].value_counts(), sep="")

carrier_group
A그룹    139504
B그룹    135205
기타      62067
Name: count, dtype: int64



수정 후:
carrier_group
B그룹        135205
A그룹         80839
기타          62067
A그룹(대형)     58665
Name: count, dtype: int64


## 13. 구간화 — pd.cut

In [15]:
f = flights.copy()
f["distance_bin"] = pd.cut(f["distance"],
                           bins=[0, 502, 872, 1389, np.inf],
                           labels=["단거리", "중거리", "장거리", "초장거리"])

print(f["distance_bin"].value_counts().sort_index())
print("\n구간별 평균 비행시간(분):")
print(f.groupby("distance_bin", observed=True)["air_time"].mean().round(1))

distance_bin
단거리     85367
중거리     84276
장거리     84375
초장거리    82758
Name: count, dtype: int64

구간별 평균 비행시간(분):
distance_bin
단거리      55.0
중거리     106.9
장거리     153.5
초장거리    286.8
Name: air_time, dtype: float64


## 14. Wide ↔ Long 변환

같은 성격의 값이 여러 열에 흩어져 있으면(wide) 그래프를 그리기 어렵다.
한 열에 값, 다른 열에 구분을 두는 long 형태로 바꾼다.

In [16]:
wide = pd.DataFrame({
    "city": ["서울", "부산", "대구"],
    "pop_2020": [966, 339, 241],
    "pop_2021": [950, 336, 239],
    "area_2020": [605.2, 770.1, 883.5],
    "area_2021": [605.2, 770.1, 883.5],
})
print("wide:\n", wide, sep="")

long = pd.wide_to_long(wide, stubnames=["pop", "area"], i="city", j="year", sep="_")
print("\nwide_to_long:\n", long, sep="")

melted = wide.melt(id_vars="city", var_name="변수", value_name="값")
print("\nmelt:\n", melted.head(6), sep="")

wide:
  city  pop_2020  pop_2021  area_2020  area_2021
0   서울       966       950      605.2      605.2
1   부산       339       336      770.1      770.1
2   대구       241       239      883.5      883.5

wide_to_long:
           pop   area
city year            
서울   2020  966  605.2
부산   2020  339  770.1
대구   2020  241  883.5
서울   2021  950  605.2
부산   2021  336  770.1
대구   2021  239  883.5

melt:
  city        변수      값
0   서울  pop_2020  966.0
1   부산  pop_2020  339.0
2   대구  pop_2020  241.0
3   서울  pop_2021  950.0
4   부산  pop_2021  336.0
5   대구  pop_2021  239.0


## 15. 결합 — merge와 concat

In [17]:
airports = pd.DataFrame({
    "origin": ["EWR", "LGA", "JFK"],
    "airport_name": ["Newark Liberty", "LaGuardia", "John F. Kennedy"],
})

sample = flights[["origin", "dest", "distance"]].head(1000)

inner = pd.merge(sample, airports, on="origin", how="inner")
left = pd.merge(sample, airports, on="origin", how="left")
print(f"inner {inner.shape}   left {left.shape}")
print(inner.head())

stacked = pd.concat([sample.head(3), sample.tail(3)], ignore_index=True)
print("\nconcat:\n", stacked, sep="")

inner (1000, 4)   left (1000, 4)
  origin dest  distance     airport_name
0    EWR  IAH      1400   Newark Liberty
1    LGA  IAH      1416        LaGuardia
2    JFK  MIA      1089  John F. Kennedy
3    JFK  BQN      1576  John F. Kennedy
4    LGA  ATL       762        LaGuardia

concat:
  origin dest  distance
0    EWR  IAH      1400
1    LGA  IAH      1416
2    JFK  MIA      1089
3    JFK  ATL       760
4    EWR  CLT       529
5    JFK  PIT       340


## 과제 1 — 출발공항별 평균 속도

1. `hour > 10` 인 항공편만 고른다
2. `speed = distance / air_time * 60` 열을 만든다 (마일/시)
3. 출발공항별 평균 속도를 구한다
4. 내림차순 정렬

In [18]:
# .copy()를 붙이는 이유 — 원본의 일부를 잘라낸 뒤 열을 추가하면
# pandas가 SettingWithCopyWarning을 낸다. 사본임을 명시하면 경고가 사라진다.
late = flights[flights["hour"] > 10].copy()
print("hour > 10:", len(late))

# air_time이 비었거나 0이면 나눗셈이 깨지므로 먼저 거른다
late = late[late["air_time"].notna() & (late["air_time"] > 0)]
late["speed"] = late["distance"] / late["air_time"] * 60

result = (late.groupby("origin")["speed"]
          .agg(평균속도="mean", 편수="count")
          .round(1)
          .sort_values("평균속도", ascending=False))
print("\n출발공항별 평균 속도 (마일/시):\n", result, sep="")

hour > 10: 221788

출발공항별 평균 속도 (마일/시):
         평균속도     편수
origin              
EWR     395.2  75182
JFK     393.9  73544
LGA     386.5  65723
